In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from typing import List, TypedDict, Dict

In [ ]:
import google.generativeai as genai
from pymilvus import Collection, connections

from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

In [ ]:
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

In [ ]:
connections.connect(alias="default", host="127.0.0.1", port="19530")
print("Milvus connected:", connections.has_connection("default"))
col = Collection(name="rag_chunks")
col.load()

In [ ]:
def embed_texts(texts: List[str], model: str = "text-embedding-004") -> List[List[float]]:
    genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
    # The google-generativeai SDK supports batching via embed_content
    # We'll call for each text for simplicity; for large corpora, batch your calls.
    embeddings = []
    for t in texts:
        resp = genai.embed_content(model=model, content=t)
        vec = resp["embedding"] if isinstance(resp, dict) else resp.embedding
        embeddings.append(vec)
    return embeddings

In [ ]:
# ===== Tool =====
@tool
def milvus_retrieve(question: str, top_k: int = 4) -> List[Dict]:
    q_emb = embed_texts([question])[0]
    search_params = {"metric_type": "COSINE", "params": {"ef": 64}}
    results = col.search(
        data=[q_emb],
        anns_field="embedding",
        param=search_params,
        limit=top_k,
        output_fields=["source", "chunk_id", "text"],
    )
    out = []
    for hits in results:
        for hit in hits:
            out.append({
                "score": float(hit.distance),
                "source": hit.entity.get("source"),
                "chunk_id": int(hit.entity.get("chunk_id")),
                "text": hit.entity.get("text"),
            })
    return out

tools = [milvus_retrieve]
tools_by_name = {t.name: t for t in tools}

In [ ]:
# ===== Model =====
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.2)
model_with_tools = llm.bind_tools(tools)

In [ ]:
# ===== State =====
class AgentState(TypedDict):
    question: str
    context: List[Document]
    chat_history: List
    answer: str

In [ ]:
# ===== Nodes =====
def llm_node(state: AgentState):
    context_ = " ".join(
        f"[Source: {d.metadata.get('source')} | Chunk: {d.metadata.get('chunk_id')}] {d.page_content}"
        for d in state["context"]
    )
    system_ = SystemMessage(
        content=(
            "You are a helpful assistant. Use ONLY the provided context. "
            "If the answer is not in context, say you don't know. Cite as [source:chunk]."
        )
    )
    messages = []
    messages.append(("system", system_))
    for m in state.get("chat_history", [])[-3:]:  # last 3 turns
        if isinstance(m, HumanMessage): messages.append(("human", m.content))
        elif isinstance(m, AIMessage): messages.append(("ai", m.content))
    messages.append(("human", f"Context: {context_} Question: {state['question']}"))

    # Convert to LC messages
    lc_msgs = []
    for role, content in messages:
        if role == "system": lc_msgs.append({"role": "system", "content": content})
        elif role == "human": lc_msgs.append(HumanMessage(content=content))
        else: lc_msgs.append(AIMessage(content=content))

    response = llm.invoke(lc_msgs)
    return {"answer": [response]}

In [ ]:
# ===== Nodes =====
def tool_node(state: AgentState):
    last_msg = state["messages"][-1]
    tool_messages = []

    for call in last_msg.tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        tool = tools_by_name[tool_name]
        tool_result = tool.invoke(tool_args)

        tool_messages.append(ToolMessage(
            content=str(tool_result),
            tool_call_id=call["id"],
        ))

    return {"messages": tool_messages}

In [ ]:
# ===== Conditional edge =====
def should_continue(state: AgentState):
    last_msg = state["messages"][-1]
    if getattr(last_msg, "tool_calls", None):
        return "tool_node"
    return END

In [ ]:
# ===== Build graph =====
workflow = StateGraph(AgentState)
workflow.add_node("llm_node", llm_node)
workflow.add_node("tool_node", tool_node)

workflow.add_edge(START, "llm_node")
workflow.add_edge("tool_node", "llm_node")
workflow.add_conditional_edges("llm_node", should_continue)

graph = workflow.compile()

In [ ]:
# ===== Conversation loop =====
chat_history: List = []

def ask(question: str) -> str:
    global chat_history
    state: AgentState = {
        "question": question,
        "context": [],
        "chat_history": chat_history,
        "answer": ""
    }
    out = graph.invoke(state)
    # update chat history
    chat_history = chat_history + [HumanMessage(content=question), AIMessage(content=out["answer"])]
    return out["answer"]

In [ ]:
print(ask('What is RAG?'))

In [ ]:
print(ask('How do LangChain and LangGraph help?'))